In [1]:
import json

import numpy as np
import pandas as pd
import photolab as lab
import torch

device = (
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)
saved = json.loads((lab.HERE / "folds.json").read_text())
files = [
    lab.TYPES / label / name for name, label in zip(saved["files"], saved["labels"], strict=True)
]
y = np.array([lab.CLASSES.index(label) for label in saved["labels"]])
fold = np.array(saved["fold"])
X = np.load(lab.CACHE / "types.npy")  # stage 6's embeddings: the head starts from the probe
probe_oof = np.load(lab.HERE / "probe_oof.npy")
probe_folds = pd.read_csv(lab.HERE / "probe_folds.csv")
print(device, "|", len(files), "photos |", fold.max() + 1, "folds")

mps | 501 photos | 5 folds


In [2]:
from PIL import Image, ImageOps
from serving import SETTINGS, preprocess
from torchvision import transforms

MEAN, STD = SETTINGS["image"]["mean"], SETTINGS["image"]["std"]
SIZE = SETTINGS["image"]["size"][0]
shake = transforms.Compose(
    [
        transforms.RandomResizedCrop(SIZE, scale=(0.6, 1.0)),  # a random part of the photo, resized
        transforms.RandomHorizontalFlip(),  # a mirror image is the same kind of photo
        transforms.ColorJitter(0.2, 0.2, 0.2),  # a bit lighter, darker, more or less colour
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ]
)


def load(path):
    return ImageOps.exif_transpose(Image.open(path)).convert("RGB")


class Photos(torch.utils.data.Dataset):
    def __init__(self, indices, train):
        self.indices, self.train = list(indices), train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        k = self.indices[i]
        x = shake(load(files[k])) if self.train else torch.from_numpy(preprocess(files[k]))
        return x, int(y[k])

In [3]:
from peft import LoraConfig, get_peft_model
from sklearn.linear_model import LogisticRegression
from transformers import SiglipVisionModel


class Classifier(torch.nn.Module):
    def __init__(self, head_weight, head_bias):
        super().__init__()
        tower = SiglipVisionModel.from_pretrained(lab.MODEL_DIR, attn_implementation="eager")
        adapters = LoraConfig(
            r=8, lora_alpha=16, lora_dropout=0.1, target_modules=["q_proj", "v_proj"]
        )
        # freezes every original weight, adds small trainable ones
        self.tower = get_peft_model(tower, adapters)
        self.head = torch.nn.Linear(head_weight.shape[1], head_weight.shape[0])
        with torch.no_grad():  # start from the probe: step one is as good as stage 6
            self.head.weight.copy_(torch.as_tensor(head_weight, dtype=torch.float32))
            self.head.bias.copy_(torch.as_tensor(head_bias, dtype=torch.float32))

    def forward(self, pixel_values):
        pooled = self.tower(pixel_values=pixel_values).pooler_output
        return self.head(torch.nn.functional.normalize(pooled, dim=-1))


def probe_start(train_idx, C):
    p = LogisticRegression(C=C, max_iter=5000).fit(X[train_idx], y[train_idx])
    return p.coef_, p.intercept_


check = Classifier(*probe_start(np.flatnonzero(fold != 0), 10.0))
check.tower.print_trainable_parameters()

/Users/rushilshandil/.cache/uv/archive-v0/82JjMR0gVF3nCFwD/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 208/208 [00:00<00:00, 14453.53it/s]
[transformers] SiglipVisionModel LOAD REPORT from: /Users/rushilshandil/dev/models/siglip-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECT

trainable params: 294,912 || all params: 93,179,136 || trainable%: 0.3165


In [4]:
EPOCHS, BATCH, LR_ADAPTERS, LR_HEAD = 6, 16, 2e-4, 1e-4


def train(train_idx, C, seed=0):
    torch.manual_seed(seed)
    model = Classifier(*probe_start(train_idx, C)).to(device)
    loader = torch.utils.data.DataLoader(
        Photos(train_idx, train=True), batch_size=BATCH, shuffle=True, num_workers=0
    )
    optimiser = torch.optim.AdamW(
        [
            {"params": [p for p in model.tower.parameters() if p.requires_grad], "lr": LR_ADAPTERS},
            {"params": model.head.parameters(), "lr": LR_HEAD},
        ],
        weight_decay=0.01,
    )
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=EPOCHS * len(loader))
    loss_fn = torch.nn.CrossEntropyLoss()
    model.train()
    for epoch in range(EPOCHS):
        total = 0.0
        for pixels, labels in loader:
            pixels, labels = pixels.to(device), labels.to(device)
            loss = loss_fn(model(pixels), labels)  # how wrong, on this batch
            optimiser.zero_grad()
            loss.backward()  # which way to nudge each trainable number
            optimiser.step()  # nudge them
            schedule.step()
            total += loss.item() * len(labels)
        print(f"    epoch {epoch + 1}: training loss {total / len(train_idx):.3f}")
    return model.eval()


@torch.no_grad()
def predict(model, indices):
    loader = torch.utils.data.DataLoader(Photos(indices, train=False), batch_size=32)
    return np.concatenate([model(x.to(device)).softmax(-1).cpu().numpy() for x, _ in loader])

In [5]:
import time

from sklearn.metrics import f1_score

lora_oof = np.empty(len(files), dtype=int)
rows = []
for k in range(fold.max() + 1):
    start = time.perf_counter()
    train_idx, test_idx = np.flatnonzero(fold != k), np.flatnonzero(fold == k)
    C = float(probe_folds.loc[probe_folds.fold == k, "C"].iloc[0])  # the same C the probe chose
    model = train(train_idx, C)
    lora_oof[test_idx] = predict(model, test_idx).argmax(1)
    rows.append(
        {
            "fold": k,
            "lora_macro_f1": f1_score(y[test_idx], lora_oof[test_idx], average="macro"),
            "minutes": (time.perf_counter() - start) / 60,
        }
    )
    print(f"fold {k}: {rows[-1]}")
lora_folds = pd.DataFrame(rows).merge(
    probe_folds[["fold", "macro_f1"]].rename(columns={"macro_f1": "probe_macro_f1"}), on="fold"
)
lora_folds["gain"] = lora_folds.lora_macro_f1 - lora_folds.probe_macro_f1
lora_folds.to_csv(lab.HERE / "lora_folds.csv", index=False)
np.save(lab.HERE / "lora_oof.npy", lora_oof)
lora_folds.round(3)

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 208/208 [00:00<00:00, 26493.83it/s]
[transformers] SiglipVisionModel LOAD REPORT from: /Users/rushilshandil/dev/models/siglip-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn

    epoch 1: training loss 0.074
    epoch 2: training loss 0.068
    epoch 3: training loss 0.051
    epoch 4: training loss 0.044
    epoch 5: training loss 0.039
    epoch 6: training loss 0.041
fold 0: {'fold': 0, 'lora_macro_f1': 0.9203763057929724, 'minutes': 1.5471152041667664}


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 208/208 [00:00<00:00, 12018.23it/s]
[transformers] SiglipVisionModel LOAD REPORT from: /Users/rushilshandil/dev/models/siglip-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn

    epoch 1: training loss 0.071
    epoch 2: training loss 0.057
    epoch 3: training loss 0.048
    epoch 4: training loss 0.038
    epoch 5: training loss 0.039
    epoch 6: training loss 0.036
fold 1: {'fold': 1, 'lora_macro_f1': 0.8714216320060774, 'minutes': 1.476160730550085}


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 208/208 [00:00<00:00, 26591.54it/s]
[transformers] SiglipVisionModel LOAD REPORT from: /Users/rushilshandil/dev/models/siglip-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn

...

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 208/208 [00:00<00:00, 26530.08it/s]
[transformers] SiglipVisionModel LOAD REPORT from: /Users/rushilshandil/dev/models/siglip-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn

    epoch 1: training loss 0.068
    epoch 2: training loss 0.056
    epoch 3: training loss 0.052
    epoch 4: training loss 0.039
    epoch 5: training loss 0.038
    epoch 6: training loss 0.033
fold 3: {'fold': 3, 'lora_macro_f1': 0.8910774468394046, 'minutes': 1.5694068361165894}


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 208/208 [00:00<00:00, 9909.53it/s]
[transformers] SiglipVisionModel LOAD REPORT from: /Users/rushilshandil/dev/models/siglip-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn

    epoch 1: training loss 0.071
    epoch 2: training loss 0.064
    epoch 3: training loss 0.045
    epoch 4: training loss 0.037
    epoch 5: training loss 0.036
    epoch 6: training loss 0.040
fold 4: {'fold': 4, 'lora_macro_f1': 0.8304667420907688, 'minutes': 1.5711296361165297}


,fold,lora_macro_f1,minutes,probe_macro_f1,gain
0,0,0.920,1.547,0.919,0.002
1,1,0.871,1.476,0.850,0.022
2,2,0.832,1.546,0.832,0.001
3,3,0.891,1.569,0.882,0.009
4,4,0.830,1.571,0.840,-0.009


In [6]:
rng = np.random.default_rng(0)
gains = []
for _ in range(2000):
    s = rng.integers(0, len(y), len(y))  # resample photos, keep each photo's two answers together
    gains.append(
        f1_score(y[s], lora_oof[s], average="macro") - f1_score(y[s], probe_oof[s], average="macro")
    )
low, high = np.percentile(gains, [2.5, 97.5])
mean_gain = f1_score(y, lora_oof, average="macro") - f1_score(y, probe_oof, average="macro")
print(
    f"LoRA minus probe, macro-F1: {mean_gain:+.3f} (95% interval {low:+.3f} to {high:+.3f}); "
    f"LoRA better on {(lora_folds.gain > 0).sum()} of {len(lora_folds)} folds"
)

LoRA minus probe, macro-F1: +0.005 (95% interval -0.008 to +0.018); LoRA better on 4 of 5 folds


In [7]:
final_C = float(probe_folds.C.mode().iloc[0])
final = train(np.arange(len(files)), final_C)
final.tower.save_pretrained(lab.HERE / "lora")  # only the adapters: a few MB
clients = pd.read_csv(lab.HERE / "client_types.csv")
client_files = [lab.PHOTOS / c / f for c, f in zip(clients.client, clients.file, strict=True)]
client_y = np.array([lab.CLASSES.index(t) for t in clients.type])
with torch.no_grad():
    client_pixels = torch.from_numpy(np.stack([preprocess(p) for p in client_files])).to(device)
    lora_client = final(client_pixels).argmax(1).cpu().numpy()
print(
    "clients, LoRA: accuracy",
    round(float((lora_client == client_y).mean()), 3),
    "| macro-F1",
    round(f1_score(client_y, lora_client, average="macro"), 3),
)

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 208/208 [00:00<00:00, 10242.86it/s]
[transformers] SiglipVisionModel LOAD REPORT from: /Users/rushilshandil/dev/models/siglip-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn

    epoch 1: training loss 0.061
    epoch 2: training loss 0.052
    epoch 3: training loss 0.044
    epoch 4: training loss 0.033
    epoch 5: training loss 0.040
    epoch 6: training loss 0.029
clients, LoRA: accuracy 0.875 | macro-F1 0.867


In [8]:
from serving import ImageEncoder, TextEncoder

text = TextEncoder()
template = SETTINGS["text"]["template"]
cases = lab.load_cases()


@torch.no_grad()
def tuned_vectors(paths):
    pixels = torch.from_numpy(np.stack([preprocess(p) for p in paths])).to(device)
    v = final.tower(pixel_values=pixels).pooler_output
    return torch.nn.functional.normalize(v, dim=-1).cpu().numpy()


original = ImageEncoder()


def searcher(embed):
    vectors = {
        c: ([p.name for p in lab.photos_of(c)], embed(lab.photos_of(c)))
        for c in {k.client for k in cases}
    }

    def search(case):
        names, photos = vectors[case.client]
        cosines = photos @ text([template.format(case.query)])[0]
        return names, cosines, float(cosines.max())

    return search


pd.DataFrame(
    {
        "original photo half": lab.evaluate(cases, searcher(original)),
        "LoRA-tuned photo half": lab.evaluate(cases, searcher(tuned_vectors)),
    }
).T.round(3)

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


,recall@1,recall@3,mrr,cases
original photo half,0.971,1.0,0.985,48.0
LoRA-tuned photo half,1.000,1.0,1.000,48.0


In [9]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(lab.MODEL_DIR)
queries = [template.format(c.query) for c in cases]
ours = text.tokenizer(queries, padding="max_length", max_length=64, return_tensors="np")
theirs = processor(text=queries, padding="max_length", max_length=64, return_tensors="np")
print("same tokens as stage 5:", bool((ours["input_ids"] == theirs["input_ids"]).all()))
search = searcher(original)
for c in cases:
    names, cosines, _ = search(c)
    top = names[int(np.argmax(cosines))]
    if c.want and top not in c.want:
        print(f"{c.client} {c.query!r}: top photo {top}, wanted {c.want}")

same tokens as stage 5: True
lone-star-roofing 'man in check shirt': top photo lone-star-roofing-08.jpg, wanted ('lone-star-roofing-04.jpg',)


In [10]:
from collections import Counter

together = text(queries)  # all 48 queries in one batch, as stage 5 did
alone = np.stack([text([q])[0] for q in queries])  # one at a time, as the server will
print("batch of 48 vs alone, worst cosine:", round(float((together * alone).sum(1).min()), 5))

COST = {"right": 0.0, "needless ask": 1.0, "wrong swap": 3.0}
scale, bias = SETTINGS["logit_scale"], SETTINGS["logit_bias"]
search = searcher(original)  # the server's way: photos fp32, one query at a time
answers = {}
for c in cases:
    names, cosines, _ = search(c)
    sure = float((1 / (1 + np.exp(-(cosines * scale + bias)))).max())
    answers[(c.client, c.query)] = (names, cosines, sure)


def outcome(case, line):
    names, cosines, sure = answers[(case.client, case.query)]
    if sure < line:
        return "needless ask" if case.want else "right"
    return "right" if names[int(np.argmax(cosines))] in case.want else "wrong swap"


def cheapest_line(group):
    lines = [0.0, *sorted({answers[(c.client, c.query)][2] for c in group}), 1.0]
    return min(lines, key=lambda t: sum(COST[outcome(c, t)] for c in group))


held_out = []
for name in sorted({c.client for c in cases}):
    line = cheapest_line([c for c in cases if c.client != name])
    held_out += [outcome(c, line) for c in cases if c.client == name]
print("one query at a time, line chosen on the other clients:", dict(Counter(held_out)))
print("line for the product:", cheapest_line(cases))

batch of 48 vs alone, worst cosine: 0.99777
one query at a time, line chosen on the other clients: {'right': 37, 'needless ask': 8, 'wrong swap': 3}
line for the product: 0.008716220036149025
